# Results

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.calibration import calibration_curve

RESULTS_DIR = '../results/'
OUTPUT_DIR = '../outputs/'

os.makedirs(OUTPUT_DIR, exist_ok=True)

COLORS = {
    'M1': 'darkorange',
    'M2': 'navy',
    'M3': 'green',
    'M4': 'red',
    'M5': 'purple',
    'M6': 'brown',
    'M7': 'pink',
    'M8': 'gray'
}

def plot_combined_roc():
    print("--- Generating Multi-Model ROC Curve ---")
    plt.figure(figsize=(10, 8))
    plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', label='Random Guess (AUC = 0.5000)')

    if not os.path.exists(RESULTS_DIR):
        print(f"Error: Directory '{RESULTS_DIR}' not found.")
        return

    json_files = sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('_results.json')])
    
    for filename in json_files:
        filepath = os.path.join(RESULTS_DIR, filename)
        with open(filepath, 'r') as f:
            data = json.load(f)
            
        model_id = data.get('model_id', filename.split('_')[0])
        y_true = np.array(data['true'])
        y_probs = np.array(data['probs'])
        auc = data.get('auc', roc_auc_score(y_true, y_probs))
        
        fpr, tpr, _ = roc_curve(y_true, y_probs)
        color = COLORS.get(model_id, None)
        
        plt.plot(fpr, tpr, color=color, lw=2, label=f'{model_id} (AUC = {auc:.4f})')

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curves\nAcross Model Configurations', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    
    output_path = os.path.join(OUTPUT_DIR, 'ablation_roc_curve.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"✅ Multi-model ROC curve saved to: {output_path}")
    plt.show()

def plot_calibration():
    print("\n--- Generating Multi-Model Calibration Curve ---")
    plt.figure(figsize=(10, 8))
    plt.plot([0, 1], [0, 1], color='black', lw=2, linestyle='--', label='Perfect Calibration')

    if not os.path.exists(RESULTS_DIR):
        return

    json_files = sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('_results.json')])

    for filename in json_files:
        filepath = os.path.join(RESULTS_DIR, filename)
        with open(filepath, 'r') as f:
            data = json.load(f)
            
        model_id = data.get('model_id', filename.split('_')[0])

        if "Proposed" in model_id:
            model_id = "M1"
            
        y_true = np.array(data['true'])
        y_probs = np.array(data['probs'])
        
        prob_true, prob_pred = calibration_curve(y_true, y_probs, n_bins=10, strategy='uniform')
        color = COLORS.get(model_id, None)
        
        plt.plot(prob_pred, prob_true, marker='s', color=color, lw=2, markersize=5, label=f'{model_id}')

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Mean Predicted Positive-Class Probability', fontsize=12)
    plt.ylabel('Observed Positive-Class Fraction', fontsize=12)
    plt.title('Reliability Diagrams Across Model Configurations', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right", fontsize=11)
    
    output_path = os.path.join(OUTPUT_DIR, 'ablation_calibration_curve.png')
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f" Calibration curve saved to: {output_path}")
    plt.show()

plot_combined_roc()
plot_calibration()